In [2]:
import litellm
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. Initialize local embedding engine
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Simulate a vector database with a mix of high-signal and noisy records
noisy_kb_data = [
    Document(page_content="MeshQuery coordination runs on port 9092.", metadata={"id": 1}),
    Document(page_content="Our office kitchen has a coffee machine on port side of the room.", metadata={"id": 2}),
    Document(page_content="The default network configuration specifies port 9092 for cluster traffic.", metadata={"id": 3}),
    Document(page_content="Make sure to export your Docker port configurations before deployment.", metadata={"id": 4}),
]

vector_store = FAISS.from_documents(noisy_kb_data, embeddings)
print("✓ Cell 1: Mock Vector DB populated with mixed signal/noise records.")

13:23:41 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
13:23:42 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
/tmp/ipykernel_121083/654133708.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✓ Cell 1: Mock Vector DB populated with mixed signal/noise records.


In [ ]:
def simple_reranker(query: str, retrieved_docs: list) -> list:
    scored_docs = []
    
    # Clean and split query into unique lowercase semantic tokens
    query_terms = set(query.lower().split())
    
    for doc in retrieved_docs:
        # Clean and split document text into lowercase tokens
        doc_terms = set(doc.page_content.lower().split())
        
        # Core Concept: Compute the raw intersection of matching terms
        matching_terms = query_terms.intersection(doc_terms)
        
        # Calculate a basic density score (Matching Terms / Total Query Terms)
        score = len(matching_terms) / len(query_terms) if query_terms else 0.0
            
        scored_docs.append((score, doc))
    
    # Sort the documents dynamically by their density score in descending order
    scored_docs.sort(key=lambda x: x[0], reverse=True)
    
    print("\n--- Dynamic Reranker Mathematical Scoring Output ---")
    print(f"Active Query Terms Evaluated: {query_terms}")
    for score, doc in scored_docs:
        print(f"Score: {score:.2f} | Text: '{doc.page_content}'")
        
    # Return only the documents, sorted by high signal
    return [doc for score, doc in scored_docs]

print("✓ Cell 2: Dynamic reranking scoring logic successfully compiled.")

✓ Cell 2: Reranking scoring logic successfully compiled.


In [4]:
def execute_reranked_rag(user_query: str):
    # Step A: Wide Net Search - Fetch 4 documents from FAISS
    initial_matches = vector_store.similarity_search(user_query, k=4)
    
    # Step B: Filter - Run them through our Reranker to filter out the noise
    ordered_matches = simple_reranker(user_query, initial_matches)
    
    # Take only the top 2 highest-scoring blocks for the context window
    top_context = "\n\n".join([doc.page_content for doc in ordered_matches[:2]])
    
    # Step C: Construct standard OpenAI-style message payload array
    messages = [
        {
            "role": "system",
            "content": f"Answer using ONLY this context:\n\n{top_context}"
        },
        {
            "role": "user", 
            "content": user_query
        }
    ]
    
    # Step D: Execute using Universal LiteLLM structure
    print(f"\nQuery: {user_query}")
    print("AI: ", end="", flush=True)
    
    response = litellm.completion(
        model="ollama/llama3.2", 
        messages=messages,
        api_base="http://localhost:11434",
        stream=True  # Standard streaming parameter flag
    )
    
    for chunk in response:
        token = chunk.choices[0].delta.content or ""
        print(token, end="", flush=True)
    print("\n")

print("✓ Cell 3: LiteLLM RAG pipeline execution block compiled.")

✓ Cell 3: LiteLLM RAG pipeline execution block compiled.


In [5]:
# Trigger the execution loop
execute_reranked_rag("What port does the cluster coordinator use?")



--- Reranker Mathematical Scoring Output ---
Score: 0.95 | Text: 'The default network configuration specifies port 9092 for cluster traffic.'
Score: 0.95 | Text: 'MeshQuery coordination runs on port 9092.'
Score: 0.40 | Text: 'Make sure to export your Docker port configurations before deployment.'
Score: 0.40 | Text: 'Our office kitchen has a coffee machine on port side of the room.'

Query: What port does the cluster coordinator use?
AI: Based on the provided information, the cluster coordinator uses port 9092.

